# Quickstart: Wavelength Selection with spectral_select

This notebook demonstrates the basic workflow for wavelength selection analysis using the `spectral_select` library.

**What you'll learn:**
- Load preprocessed hyperspectral data
- Configure and run wavelength selection analysis
- Extract selected wavelength bands
- Create basic visualizations

## 1. Setup

Import the core classes from `spectral_select`:

In [ ]:
from spectral_select import Analyzer, Config, SpectraData, Visualizer

## 2. Load Data

Load preprocessed hyperspectral data from a pickle file. The data should have been preprocessed using the standard pipeline.

In [ ]:
from pathlib import Path

# Get project root (assuming notebook is in notebooks/examples/)
project_root = Path.cwd().parent.parent

# Data path - update this for your data file
DATA_PATH = project_root / "Data" / "processed" / "LichensProcessed" / "data_cutoff_60nm_exposure_max_power_min.pkl"

# Verify data file exists
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}\nRun 00_data_loading.ipynb first to process raw data.")

data = SpectraData.from_pickle(DATA_PATH)

# SpectraData is a multi-excitation container
print(f"Sample: {data.sample_name}")
print(f"Spatial shape: {data.spatial_shape}")
print(f"Number of excitations: {data.n_excitations}")
print(f"Excitation wavelengths: {data.excitation_wavelengths}")

# Access per-excitation data
print("\nPer-excitation details:")
for ex_nm in data.excitation_wavelengths:
    ex_data = data.get_excitation(ex_nm)
    print(f"  Ex {ex_nm:.0f}nm: {ex_data.n_bands} bands, shape {ex_data.shape}")

## 3. Configure Analysis

Create a configuration with your analysis parameters. The `Config` class provides sensible defaults for most parameters.

In [ ]:
config = Config(
    sample_name="LichensProcessed",
    n_bands_to_select=10,
    device="cpu",  # Use "cuda" if GPU available
    training_epochs=20,  # Use fewer epochs for faster execution
)

print(f"Sample: {config.sample_name}")
print(f"Bands to select: {config.n_bands_to_select}")
print(f"Device: {config.device}")
print(f"Training epochs: {config.training_epochs}")

## 4. Run Analysis

Create an `Analyzer` and fit it to the data. This runs the full wavelength selection pipeline:
1. Train autoencoder on spectral data
2. Extract baseline representations
3. Perturb input dimensions and measure influence
4. Rank wavelengths by influence score

In [ ]:
analyzer = Analyzer(config)
analyzer.fit(data)

print("Analysis complete!")
print(f"Selected bands: {analyzer.result.n_bands}")
print(f"Compression ratio: {analyzer.result.metrics.compression_ratio:.1f}x")

## 5. Get Results

Extract the selected wavelength bands. Each band includes excitation/emission wavelengths, influence score, and rank.

In [ ]:
bands = analyzer.get_wavelengths()

print(f"Selected {len(bands)} wavelength bands:\n")

# Show top 10 bands
for band in bands[:10]:
    print(f"Rank {band.rank:2d}: Ex={band.excitation_nm:3.0f}nm, Em={band.emission_nm:3.0f}nm, Score={band.influence_score:.4f}")

## 6. Visualize Results

Create visualizations using the `Visualizer` class. The factory method `from_analyzer` automatically extracts the needed data.

In [ ]:
viz = Visualizer.from_analyzer(analyzer)

In [ ]:
# Influence heatmap shows score distribution across excitation/emission grid
viz.plot_influence_heatmap()

In [ ]:
# Scatter plot shows selected wavelengths with size indicating rank
viz.plot_wavelength_scatter()

In [ ]:
# Ranking bar chart shows influence scores by rank
viz.plot_influence_ranking()

## Summary

In this notebook, you learned how to:

1. **Load data** with `SpectraData.from_pickle()`
2. **Configure** analysis with `Config()`
3. **Run analysis** with `Analyzer.fit()`
4. **Extract results** with `Analyzer.get_wavelengths()`
5. **Visualize** with `Visualizer.from_analyzer()`

**Next steps:**
- See `02_validation.ipynb` for ground truth validation
- Explore different `Config` parameters
- Try different datasets